In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip -q install transformers datasets evaluate peft accelerate scikit-learn sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.2 MB/s eta 0:00:00


In [3]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)

from peft import (
    PromptTuningConfig,
    PromptTuningInit,
    TaskType,
    get_peft_model,
)

from sklearn.metrics import classification_report, confusion_matrix

In [4]:
seed = 42
set_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [5]:
dataset = load_dataset("glue", "qnli")

print(dataset)
print(dataset["train"][0])
print(dataset["validation"][0])

README.md: 0.00B [00:00, ?B/s]

qnli/train-00000-of-00001.parquet:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

qnli/validation-00000-of-00001.parquet:   0%|          | 0.00/872k [00:00<?, ?B/s]

qnli/test-00000-of-00001.parquet:   0%|          | 0.00/877k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/104743 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5463 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5463 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
})
{'question': 'When did the third Digimon series begin?', 'sentence': 'Unlike the two seasons before it and most of the seasons that followed, Digimon Tamers takes a darker and more realistic approach to its story featuring Digimon who do not reincarnate after their deaths and more complex character development in the original Japanese.', 'label': 1, 'idx': 0}
{'question': 'What came into force after the new constitution was herald?', 'sentence': 'As of that day, the new constitution heralding the Second Republic came into force.', 'label': 0, 'idx': 0}


In [6]:
model_name = "t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
base_model.to(device)

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [7]:
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    prompt_tuning_init=PromptTuningInit.RANDOM,
    num_virtual_tokens=20,
    tokenizer_name_or_path=model_name,
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()
model.to(device)

trainable params: 30,720 || all params: 222,934,272 || trainable%: 0.0138


PeftModelForSeq2SeqLM(
  (base_model): T5ForConditionalGeneration(
    (shared): Embedding(32128, 768)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 768)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=768, out_features=768, bias=False)
                (k): Linear(in_features=768, out_features=768, bias=False)
                (v): Linear(in_features=768, out_features=768, bias=False)
                (o): Linear(in_features=768, out_features=768, bias=False)
                (relative_attention_bias): Embedding(32, 12)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=768, out_features=3072, bias=False)
                (wo): Li

In [8]:
label_map = {
    0: "entailment",
    1: "not_entailment",
}

max_input_length = 256
max_target_length = 8

def preprocess_function(examples):
    inputs = []
    targets = []

    for question, sentence, label in zip(
        examples["question"],
        examples["sentence"],
        examples["label"]
    ):
        text = f"question: {question} sentence: {sentence} Does the sentence answer the question?"
        target = label_map[int(label)]

        inputs.append(text)
        targets.append(target)

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )

    label_tokens = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length",
    )

    labels = []
    for seq in label_tokens["input_ids"]:
        labels.append([
            token if token != tokenizer.pad_token_id else -100
            for token in seq
        ])

    model_inputs["labels"] = labels
    return model_inputs

In [9]:
encoded_train = dataset["train"].map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
    load_from_cache_file=False,
    desc="Tokenizing train"
)

encoded_validation = dataset["validation"].map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["validation"].column_names,
    load_from_cache_file=False,
    desc="Tokenizing validation"
)

Tokenizing train:   0%|          | 0/104743 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/5463 [00:00<?, ? examples/s]

In [10]:
sample = encoded_train[0]

print(sample.keys())
print("input_ids type:", type(sample["input_ids"]), "len:", len(sample["input_ids"]))
print("attention_mask type:", type(sample["attention_mask"]), "len:", len(sample["attention_mask"]))
print("labels type:", type(sample["labels"]), "len:", len(sample["labels"]))

print("First 20 input_ids:", sample["input_ids"][:20])
print("Labels:", sample["labels"])

dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids type: <class 'list'> len: 256
attention_mask type: <class 'list'> len: 256
labels type: <class 'list'> len: 8
First 20 input_ids: [822, 10, 366, 410, 8, 1025, 21951, 2157, 939, 1731, 58, 7142, 10, 3, 8739, 8, 192, 9385, 274, 34]
Labels: [59, 834, 35, 5756, 297, 1, -100, -100]


In [11]:
encoded_train.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

encoded_validation.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [12]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_base_prompt_qnli",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=5e-4,
    num_train_epochs=3,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=8,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    load_best_model_at_end=False,
)

In [14]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    label_to_id = {
        "entailment": 0,
        "not_entailment": 1,
    }

    y_pred = [label_to_id.get(x.strip(), 1) for x in decoded_preds]
    y_true = [label_to_id.get(x.strip(), 1) for x in decoded_labels]

    accuracy = (np.array(y_pred) == np.array(y_true)).mean()

    return {"accuracy": accuracy}

In [15]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_validation,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [16]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.121532,0.035729,0.927512
2,0.089789,0.036611,0.927329
3,0.092579,0.036635,0.927512


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=19641, training_loss=0.18723758714795447, metrics={'train_runtime': 11843.8033, 'train_samples_per_second': 26.531, 'train_steps_per_second': 1.658, 'total_flos': 9.567611449638912e+16, 'train_loss': 0.18723758714795447, 'epoch': 3.0})

In [17]:
eval_results = trainer.evaluate()
print(eval_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.03663506731390953, 'eval_accuracy': 0.9275123558484349, 'eval_runtime': 206.3327, 'eval_samples_per_second': 26.477, 'eval_steps_per_second': 0.829, 'epoch': 3.0}


In [18]:
pred_output = trainer.predict(encoded_validation)

preds = pred_output.predictions
labels = pred_output.label_ids

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

label_to_id = {
    "entailment": 0,
    "not_entailment": 1,
}

y_pred = [label_to_id.get(x.strip(), 1) for x in decoded_preds]
y_true = [label_to_id.get(x.strip(), 1) for x in decoded_labels]

print("Classification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=["entailment", "not_entailment"],
    digits=4
))

print("Confusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Classification Report:

                precision    recall  f1-score   support

    entailment     0.9168    0.9386    0.9276      2702
not_entailment     0.9385    0.9167    0.9274      2761

      accuracy                         0.9275      5463
     macro avg     0.9276    0.9276    0.9275      5463
  weighted avg     0.9278    0.9275    0.9275      5463

Confusion Matrix:

[[2536  166]
 [ 230 2531]]
